# relpath.dev — Quickstart

Connect a local database, ask a predictive question in plain language, and get an **explained** answer — local-first (nothing leaves your machine).

In [1]:
import warnings
warnings.filterwarnings('ignore')
import relpath as rp
rp.__version__

'0.1.0'

## 1. Build the bundled synthetic e-commerce DB (zero downloads) and connect

In [2]:
import os, tempfile
from relpath.sample_data import build
db = os.path.join(tempfile.gettempdir(), 'relpath_quickstart.duckdb')
build(db)
engine = rp.connect(db)
print(engine.describe())

Wrote C:\Users\isiko\AppData\Local\Temp\relpath_quickstart.duckdb
  customers=1200  products=220  transactions=14343  returns=2209
Tables:
  customers (pk=customer_id, time=signup_date) [customer_id:pk, signup_date:time, country:categorical, segment:categorical]
  products (pk=product_id, time=None) [product_id:pk, category:categorical, brand:categorical, price:numeric]
  returns (pk=return_id, time=return_time) [return_id:pk, tx_id:fk, return_time:time, reason:categorical]
  transactions (pk=tx_id, time=tx_time) [tx_id:pk, customer_id:fk, product_id:fk, tx_time:time, quantity:numeric, amount:numeric]
Foreign keys:
  returns.tx_id -> transactions.tx_id
  transactions.customer_id -> customers.customer_id
  transactions.product_id -> products.product_id


## 2. Ask in plain language → PQL
Uses Claude if `ANTHROPIC_API_KEY` is set, otherwise the offline template matcher.

In [3]:
nl = engine.ask('gelecek 30 gunde islem yapmayacak musteriler')
print(nl.source, '->', nl.pql)

template -> PREDICT COUNT(transactions.*, 0, 30, days) == 0 FOR EACH customers.customer_id


## 3. Predict (PQL or natural language both work)

In [4]:
result = engine.predict(nl.pql)
result.metrics

{'roc_auc': 0.7491971098938128, 'accuracy': 0.7366666666666667}

In [5]:
result.top(5)

,customer_id,score,label
8,9,0.997148,1
213,214,0.996367,1
1115,1116,0.996171,1
623,624,0.995570,1
1118,1119,0.995523,1


## 4. Explain — global drivers and a single entity (join-path provenance)

In [6]:
_ = result.explain(top_n=5)

En etkili 5 surucu (global, gain):
  * MONTH MONTH(signup_date)           MONTH(signup_date)
  * MEAN transactions -> products     MEAN over transactions, products  ·  MEAN(transactions.products.price)
  * STD transactions -> products     STD over transactions, products  ·  STD(transactions.products.price)
  * MIN transactions                 MIN over transactions  ·  MIN(transactions.amount)
  * SUM transactions                 SUM over transactions  ·  SUM(transactions.amount)


In [7]:
top_id = result.top(1).iloc[0][result.entity_key]
_ = result.explain(entity_id=top_id, top_n=5)

entity 9.0 -- probability 0.997
  |- MONTH(signup_date): MONTH = 4   katki +1.057
  |- transactions -> products: STD = 50.8259   katki +1.038
  |- transactions: SUM = 460.23   katki +0.436
  |- transactions: NUM_UNIQUE = 3   katki +0.385
  `- returns -> transactions: MEAN = 148.19   katki +0.331


## 5. Demand forecast (regression template)

In [8]:
fc = engine.forecast(entity='products', event='transactions', column='quantity', horizon_months=3)
fc.metrics

{'mae': 8.40608691628648, 'rmse': 10.098720355076283}